# Lanzador: visualización de una crisis de cada tipo

Ejecuta `03v_visualizacion_crisis.ipynb` **una vez por cada tipo de crisis** y guarda cada ejecución en su carpeta:

```
outputs/fase_01/03_visualizacion/<tipo>/
    03v_visualizacion_<tipo>.ipynb  <- el notebook ejecutado, con todas sus salidas
    figuras/                        <- las figuras de ese tipo
    configuracion.json, crisis_ejemplo.csv
```

Así solo hay **un notebook que mantener** (el 03v): si se cambia, se vuelve a ejecutar este lanzador y se regeneran todos los tipos.

**Cómo funciona.** Usa **papermill**, que ejecuta un notebook con parámetros:

1. En el 03v, la celda `TIPO_CRISIS = 'absz'` está marcada como celda de **parámetros** (etiqueta `parameters`).
2. Papermill inserta debajo una celda con el valor que le pasamos (`TIPO_CRISIS = 'fnsz'`, por ejemplo), ejecuta el notebook entero y guarda la copia ejecutada. El original no se toca.
3. Si un tipo falla, papermill guarda igualmente el notebook hasta la celda del error, y el lanzador sigue con el siguiente tipo.

**Requisito**: instalar papermill una vez en el entorno: `pip install papermill`.

## 1. Importación de librerías

- `papermill`: ejecuta notebooks con parámetros.
- `tfm_comun`: rutas, iconos y la descripción de cada tipo de crisis (`tipos_crisis.py`). Las primeras líneas añaden la carpeta `ALMA` al path de Python, porque este notebook está en `fase_01/`.

In [1]:
import sys
from pathlib import Path

# tfm_comun está en ALMA, por encima de la carpeta de la fase: lo añadimos al path de Python
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tfm_comun').is_dir())))

import json
import time

import pandas as pd
import papermill as pm
from IPython.display import display

from tfm_comun.rutas import RUTA_ALMA, RUTA_FASE_01, RUTA_OUTPUTS_FASE_01
from tfm_comun.iconos import con_icono, tabla_con_iconos
from tfm_comun.tipos_crisis import TIPOS_CRISIS

print(f'papermill {pm.__version__}')

papermill 2.7.0


## 2. Configuración

- `TIPOS`: tipos de crisis a visualizar. Cada uno tiene que estar descrito en `tfm_comun/tipos_crisis.py`.
  - 🔳 `mysz` (mioclónica) **queda fuera**: solo hay 2 crisis en 🔵 train, del mismo paciente, y ninguna con lead-in ≥ 5 min.
  - `cnsz` y `atsz` no aparecen en TUSZ v2.0.6.
- `NOTEBOOK`: el notebook que se ejecuta.

In [2]:
TIPOS = ['absz', 'tnsz', 'fnsz', 'gnsz', 'cpsz', 'tcsz', 'spsz']
NOTEBOOK = RUTA_FASE_01 / '03v_visualizacion_crisis.ipynb'

sin_describir = [t for t in TIPOS if t not in TIPOS_CRISIS]
assert not sin_describir, f'Faltan en tfm_comun/tipos_crisis.py: {sin_describir}'
assert NOTEBOOK.exists(), f'No encuentro {NOTEBOOK}'
print('Tipos:', '  '.join(con_icono(t) for t in TIPOS))

Tipos: 🟩 absz  🟨 tnsz  🟥 fnsz  🟦 gnsz  🟧 cpsz  🟪 tcsz  🟫 spsz


## 3. Ejecución

Un tipo tarda alrededor de un minuto. La barra de progreso de papermill muestra por qué celda va.

In [3]:
resumen = []
for tipo in TIPOS:
    carpeta = RUTA_OUTPUTS_FASE_01 / '03_visualizacion' / tipo
    carpeta.mkdir(parents=True, exist_ok=True)
    salida = carpeta / f'03v_visualizacion_{tipo}.ipynb'
    print(f'\n{con_icono(tipo)} -> {salida.relative_to(RUTA_ALMA)}')
    inicio = time.time()
    try:
        pm.execute_notebook(str(NOTEBOOK), str(salida), parameters={'TIPO_CRISIS': tipo},
                            cwd=str(RUTA_FASE_01), progress_bar=True)
        estado = '✅ ok'
    except pm.PapermillExecutionError as error:
        # El notebook queda guardado hasta la celda que falló; seguimos con el siguiente tipo
        estado = f'🔴 error en la celda {error.exec_count}: {error.ename}'
    configuracion = carpeta / 'configuracion.json'
    ejemplo = json.loads(configuracion.read_text(encoding='utf-8')).get('ejemplo', '-') if configuracion.exists() else '-'
    resumen.append({'tipo': tipo, 'ejemplo': ejemplo, 'estado': estado,
                    'segundos': round(time.time() - inicio),
                    'figuras': len(list((carpeta / 'figuras').glob('*.png')))})

D:\Dani\TFM IA\03.Codigo\ALMA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



🟩 absz -> outputs\fase_01\03_visualizacion\absz\03v_visualizacion_absz.ipynb


Executing: 100%|██████████| 57/57 [00:17<00:00,  3.33cell/s]



🟨 tnsz -> outputs\fase_01\03_visualizacion\tnsz\03v_visualizacion_tnsz.ipynb


Executing: 100%|██████████| 57/57 [00:16<00:00,  3.50cell/s]



🟥 fnsz -> outputs\fase_01\03_visualizacion\fnsz\03v_visualizacion_fnsz.ipynb


... (5 outputs left out)


🟪 tcsz -> outputs\fase_01\03_visualizacion\tcsz\03v_visualizacion_tcsz.ipynb


Executing: 100%|██████████| 57/57 [00:16<00:00,  3.55cell/s]



🟫 spsz -> outputs\fase_01\03_visualizacion\spsz\03v_visualizacion_spsz.ipynb


Executing: 100%|██████████| 57/57 [00:20<00:00,  2.84cell/s]


## 4. Resumen

Una fila por tipo: el fichero de ejemplo que eligió el notebook, si terminó bien y cuántas figuras generó. Para ver los resultados de un tipo, abre `outputs/fase_01/03_visualizacion/<tipo>/03v_visualizacion_<tipo>.ipynb`.

In [4]:
resumen = pd.DataFrame(resumen).set_index('tipo')
display(tabla_con_iconos(resumen))
resumen.to_csv(RUTA_OUTPUTS_FASE_01 / '03_visualizacion' / 'resumen_ejecuciones.csv')

,ejemplo,estado,segundos,figuras
tipo,,,,
🟩 absz,aaaaacrb_s001_t001,✅ ok,17,16
🟨 tnsz,aaaaanjw_s001_t000,✅ ok,16,16
🟥 fnsz,aaaaardk_s002_t002,✅ ok,18,16
🟦 gnsz,aaaaaacz_s006_t000,✅ ok,28,16
🟧 cpsz,aaaaamtj_s001_t000,✅ ok,21,16
🟪 tcsz,aaaaabdv_s001_t001,✅ ok,16,16
🟫 spsz,aaaaampz_s004_t002,✅ ok,20,16
